In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"


In [0]:
# 1. Read Bronze raw stream
df_bronze = spark.read.format("delta").load(bronze_table_path)

# 2. Clean & Deduplicate Intra-Batch
# - Filter corrupted records where rescued data is present
# - Window by primary key (event_id) ordered by event_time descending to keep latest state per batch
window_spec = Window.partitionBy("event_id").orderBy(F.col("event_time").desc())

df_stage = (
    df_bronze
    .filter(F.col("_rescued_data").isNull() & F.col("event_id").isNotNull())
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_rescued_data")
    .withColumn("_updated_at", F.current_timestamp())
)

# 3. Initialize Silver Table if non-existent
if not DeltaTable.isDeltaTable(spark, silver_scd1_path):
    (
        df_stage.limit(0)
        .write.format("delta")
        .mode("overwrite")
        .save(silver_scd1_path)
    )

# 4. Perform SCD Type 1 Upsert via Delta MERGE
# - Match condition: silver.event_id = stage.event_id
# - Update condition: only update when incoming event_time is newer or equal (prevents out-of-order stale overwrites)
# - Insert condition: insert new primary keys
target_table = DeltaTable.forPath(spark, silver_scd1_path)

(
    target_table.alias("silver")
    .merge(
        df_stage.alias("stage"),
        "silver.event_id = stage.event_id"
    )
    .whenMatchedUpdate(
        condition="stage.event_time >= silver.event_time",
        set={
            "title": "stage.title",
            "magnitude": "stage.magnitude",
            "place": "stage.place",
            "event_time": "stage.event_time",
            "longitude": "stage.longitude",
            "latitude": "stage.latitude",
            "depth_km": "stage.depth_km",
            "tsunami_flag": "stage.tsunami_flag",
            "_ingested_at": "stage._ingested_at",
            "_source_file": "stage._source_file",
            "_updated_at": "stage._updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"
df_silver = spark.read.format("delta").load(silver_scd1_path)

total_count = df_silver.count()
distinct_keys = df_silver.select("event_id").distinct().count()

print("=== SILVER SCD TYPE 1 VERIFICATION ===")
print(f"Total Silver Records: {total_count}")
print(f"Distinct Primary Keys (event_id): {distinct_keys}")
print(f"Deduplication Status: {'PASSED' if total_count == distinct_keys else 'FAILED'}")

df_silver.select(
    "event_id", "magnitude", "depth_km", "event_time", "_updated_at"
).show(5, truncate=False)